<a href="https://colab.research.google.com/github/SavindiPanditha/SE4050-Brain-Tumor-Classification/blob/feature%2Fefficientnetb0/notebooks/04_EfficientNetB0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EfficientNetB0 - Brain Tumor MRI Classification

This notebook implements an EfficientNetB0 model for supervised classification of brain MRI images into four classes:

- 0 - Glioma
- 1 - Meningioma
- 2 - No Tumor
- 3 - Pituitary

The experiment follows the common dataset splits and training settings defined for the group project.

## 1. Environment and Configuration

In [ ]:
# Check the environment and set reproducibility

import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from PIL import Image
from IPython.display import display

SEED = 42

tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Seed:", SEED)

In [ ]:
# Connect to Google Drive

from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Set dataset paths

DATASET_PATH = "/content/drive/MyDrive/SE4050_DL_Assignment/Dataset/dataset"
SPLIT_PATH = os.path.join(DATASET_PATH, "splits")

print("Dataset path:", DATASET_PATH)
print("Split path:", SPLIT_PATH)

In [ ]:
# Set common training settings

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 10
NUM_CLASSES = 4

CLASS_NAMES = [
    "Glioma",
    "Meningioma",
    "No Tumor",
    "Pituitary"
]

print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Seed:", SEED)
print("Classes:", CLASS_NAMES)

## 2. Dataset and Data Splits

In [ ]:
# Load and verify the common train, validation, and test splits

train_df = pd.read_csv(os.path.join(SPLIT_PATH, "train.csv"))
val_df = pd.read_csv(os.path.join(SPLIT_PATH, "val.csv"))
test_df = pd.read_csv(os.path.join(SPLIT_PATH, "test.csv"))

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nColumns:", train_df.columns.tolist())

print("\nTraining class distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation class distribution:")
print(val_df["label"].value_counts().sort_index())

print("\nTest class distribution:")
print(test_df["label"].value_counts().sort_index())

In [ ]:
# Verify that the CSV paths point to real MRI images

sample_relative_path = train_df.iloc[0]["filepath"]
sample_full_path = os.path.join(DATASET_PATH, sample_relative_path)

print("Relative path:", sample_relative_path)
print("Full path:", sample_full_path)
print("File exists:", os.path.exists(sample_full_path))

img = Image.open(sample_full_path)

print("Image size:", img.size)
print("Image mode:", img.mode)

display(img)

## 3. Image Preprocessing

In [ ]:
# Define the image loading function

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)

    return image, label

In [ ]:
# Create TensorFlow datasets from the CSV splits

def create_dataset(df, shuffle=False):
    paths = [
        os.path.join(DATASET_PATH, path)
        for path in df["filepath"].astype(str)
    ]

    labels = df["label"].astype(np.int32).values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        ds = ds.shuffle(
            len(df),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


train_ds = create_dataset(train_df, shuffle=True)
val_ds = create_dataset(val_df, shuffle=False)
test_ds = create_dataset(test_df, shuffle=False)

print("Datasets created successfully.")

In [ ]:
# Check the shape of one batch

images, labels = next(iter(train_ds))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image data type:", images.dtype)
print("Label data type:", labels.dtype)

## 4. EfficientNetB0 Model

EfficientNetB0 is used as a pretrained feature extractor with ImageNet weights. The pretrained backbone is kept frozen during the initial training phase, and a new classification head is added for the four brain tumor classes.

In [ ]:
# Import EfficientNetB0 and model-building layers

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0

In [ ]:
# Load the pretrained EfficientNetB0 backbone

base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(*IMG_SIZE, 3)
)

base_model.trainable = False

print("Base model:", base_model.name)
print("Trainable:", base_model.trainable)

In [ ]:
# Build the EfficientNetB0 classification model

inputs = keras.Input(shape=(*IMG_SIZE, 3))

x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

model = keras.Model(inputs, outputs)

model.summary()

In [ ]:
# Compile the model

model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

print("Model compiled successfully.")

In [ ]:
# Verify the model configuration

print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)
print("Trainable parameters:", model.count_params() - base_model.count_params())
print("Total parameters:", model.count_params())

## 5. Model Training
The EfficientNetB0 model is trained using the shared training and validation splits. The pretrained backbone remains frozen during this baseline experiment.

In [ ]:
# Train the EfficientNetB0 model

start_time = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

training_time = time.time() - start_time

print(f"\nTraining time: {training_time / 60:.2f} minutes")

## 6. Training Results

In [ ]:
# Display the training history

history_df = pd.DataFrame(history.history)

print(history_df)

In [ ]:
# Plot training and validation accuracy

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("EfficientNetB0 Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Plot training and validation loss

plt.figure(figsize=(8, 5))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("EfficientNetB0 Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()